In [ ]:
# Multilayer perceptron (MLP)

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import matplotlib.pyplot as plt

from notebooks.utils import display

import torch
import torch.nn as nn
from torchvision import datasets, transforms

import torch.nn.functional as F
from torch.utils.data import DataLoader

from pathlib import Path

In [ ]:
NUM_CLASSES = 10

In [ ]:
data_path = Path("../../data")
transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = datasets.CIFAR10(
    root=str(data_path),
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root=str(data_path),
    train=False,
    download=True,
    transform=transform
)

In [ ]:

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = nn.Linear(32 * 32 * 3, 200)
        self.relu1 = nn.ReLU()
        self.linear2 = nn.Linear(200, 150)
        self.relu2 = nn.ReLU()
        self.linear3 = nn.Linear(150, NUM_CLASSES)
        self.softmax = nn.Softmax()
    def forward(self, x):
        x = self.linear1(x)
        x = self.relu1(x)
        x = self.linear2(x)
        x = self.relu2(x)
        x = self.linear3(x)
        # x = self.softmax(x)
        return x

model = MLP()    
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optim = torch.optim.Adam(model.parameters(), lr=0.001)


In [ ]:
model.train()
epochs = 10
for epoch in range(epochs):
    for i, (inputs, targets) in enumerate(train_loader):
        inputs = inputs.to(device)
        inputs = inputs.reshape(inputs.shape[0], -1)
        targets = targets.to(device)

        optim.zero_grad()
        preds = model(inputs)
        loss = loss_fn(preds, targets)
        loss.backward()
        optim.step()
        if i % 10 == 0:
            print(f"{epoch}, {i}, loss: {loss.item()}")


In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score


def evaluate_with_sklearn(model, val_loader, device):
    model.eval()
    all_preds, all_targets = [], []

    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            inputs = inputs.reshape(inputs.shape[0], -1)

            outputs = model(inputs)


            preds = torch.argmax(outputs, dim=1)

            all_preds.append(preds.cpu())
            all_targets.append(targets.cpu())

    all_preds = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()

    f1 = f1_score(all_targets, all_preds, average="weighted")
    pr = precision_score(all_targets, all_preds, average="weighted")
    rec = recall_score(all_targets, all_preds, average="weighted")
    acc = accuracy_score(all_targets, all_preds)

    return f1, pr, rec, acc

In [ ]:
f1, precision, recall, accuracy = evaluate_with_sklearn(model, test_loader, device)
print("accuracy", accuracy)
print("recall", recall)
print("precision", precision)
print("f1", f1)

In [ ]:
CLASSES = np.array(
    [
        "airplane",
        "automobile",
        "bird",
        "cat",
        "deer",
        "dog",
        "frog",
        "horse",
        "ship",
        "truck",
    ]
)


model.eval()
all_preds, all_targets = [], []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        inputs = inputs.reshape(inputs.shape[0], -1)

        outputs = model(inputs)
        preds = torch.argmax(outputs, dim=1)

        all_preds.append(preds.cpu())
        all_targets.append(targets.cpu())

all_preds = torch.cat(all_preds).numpy()
all_targets = torch.cat(all_targets).numpy()
preds_single = CLASSES[all_preds]
actual_single = CLASSES[all_targets]


In [ ]:
n_to_show = 10
indices = np.random.choice(range(len(test_dataset)), n_to_show)

fig = plt.figure(figsize=(15, 3))
fig.subplots_adjust(hspace=0.4, wspace=0.4)

for i, idx in enumerate(indices):
    img, _ = test_dataset[idx]
    img = transforms.ToPILImage()(img)
    ax = fig.add_subplot(1, n_to_show, i + 1)
    ax.axis("off")
    ax.text(
        0.5,
        -0.35,
        "pred = " + str(preds_single[idx]),
        fontsize=10,
        ha="center",
        transform=ax.transAxes,
    )
    ax.text(
        0.5,
        -0.7,
        "act = " + str(actual_single[idx]),
        fontsize=10,
        ha="center",
        transform=ax.transAxes,
    )
    ax.imshow(img)